In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set seed 
np.random.seed(42)

In [ ]:
sns.set_theme(style="whitegrid")
sns.set_context("paper", font_scale=1.2)
sns.set_palette("Spectral", n_colors=12)

sns.color_palette("Spectral", n_colors=12)


# 1. Handling the data

In [ ]:
df = pd.read_csv('forPython.csv', sep = ';', decimal = ',')
df.head()

In [ ]:
# Drop columns where sample is US 
df = df[df['Sample'] != 'US']
df.head()

In [ ]:
df[df["Time"] == "Day 1"]

In [ ]:
df_day1 = df[df["Time"] == "Day 1"]
df_day1_cDC1_nochange = df_day1[df_day1["Sample"] == "cDC1 no change"]
df_day1_cDC1_change_everyday = df_day1[df_day1["Sample"] == "cDC1 change eveyday"]
df_day1_HSC_nochange = df_day1[df_day1["Sample"] == "HSC no change"]
df_day1_HSC_change1of2 = df_day1[df_day1["Sample"] == "HSC change 1 of 2"]


# 2. Preprocessing

In [ ]:
def split_by_day (dataframe):
    sample_days = {}
    for day in dataframe["Time"].unique():
        sample_days[day] = dataframe[dataframe["Time"] == day]
    return sample_days

In [ ]:
def split_by_sample(df):
    sample_dict = {}
    for sample_name in df["Sample"].unique():
        sample_dict[sample_name] = df[df["Sample"] == sample_name]

    return sample_dict


In [ ]:
results_by_day = split_by_day(df)

# 3. Figures

In [ ]:
# Plot a figure with the 6 days with 3 lines and 2 columns
fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(12, 20))
days = list(results_by_day.keys())
for i, ax in enumerate(axes.flatten()):
    if i < len(days):
        day = days[i]
        day_data = results_by_day[day]
        day_proportions = pd.DataFrame()
        for sample_name, sample_data in split_by_sample(day_data).items():
            sample_proportions = sample_data.iloc[:, 2:].copy()
            sample_proportions.insert(0, "Sample", sample_name)
            day_proportions = pd.concat([day_proportions, sample_proportions], ignore_index=True)
        
        day_proportions_plot = day_proportions.melt(id_vars='Sample', var_name='Cell Type', value_name='Proportion')
        day_proportions_plot_pivot = day_proportions_plot.pivot(index='Sample', columns='Cell Type', values='Proportion')
        day_proportions_plot_pivot.plot(kind='bar', stacked=True, ax=ax)
        ax.set_title(f'Cell Type Proportions for {day} Samples')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.legend_.remove()
plt.tight_layout()

plt.legend(
    loc='upper center',
    bbox_to_anchor=(-0.2, -0.5),
    ncol =6,
    frameon=False
)
plt.savefig('Figures/DCP_proportions_across_days.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
dcp_proportions = df.pivot(index='Time', columns='Sample', values='DCPs')
dcp_proportions

In [ ]:
# Merge columns two columns "HSC change 1 of 2 
dcp_proportions['HSC change 1 of 2 new'] = dcp_proportions[['HSC change 1 of 2', 'HSC change 1 of 2 ']].sum(axis=1)
dcp_proportions = dcp_proportions.drop(columns=['HSC change 1 of 2', 'HSC change 1 of 2 '])
dcp_proportions["HSC change 1 of 2"] = dcp_proportions['HSC change 1 of 2 new']
dcp_proportions.drop(columns=['HSC change 1 of 2 new'], inplace=True)


In [ ]:
# Merge columns two columns "cDC1 change 1 of 2
dcp_proportions['cDC1 change 1 of 2 new'] = dcp_proportions[['cDC1 change 1 of 2', 'cDC1 change 1 of 2 ']].sum(axis=1)
dcp_proportions = dcp_proportions.drop(columns=['cDC1 change 1 of 2', 'cDC1 change 1 of 2 '])
dcp_proportions["cDC1 change 1 of 2"] = dcp_proportions['cDC1 change 1 of 2 new']
dcp_proportions.drop(columns=['cDC1 change 1 of 2 new'], inplace=True)

dcp_proportions

In [ ]:
dcp_proportions['HSC no change new'] = dcp_proportions[['HSC no change', 'HSC change eveyday']].sum(axis=1)
dcp_proportions = dcp_proportions.drop(columns=['HSC no change', 'HSC change eveyday'])
dcp_proportions["HSC no change"] = dcp_proportions['HSC no change new']
dcp_proportions.drop(columns=['HSC no change new'], inplace=True)

dcp_proportions

In [ ]:
df_before_thawing = df[df["Time"] == "Before Thawing"]
df_day0 = df[df["Time"] == "Day 0"]

# Add df_day 0 after df_before thawing
df_1st_days = pd.concat([df_before_thawing, df_day0], ignore_index=True)
df_1st_days

In [ ]:
# Plot bar chart of DCP proportions on first days
df_1st_days["DCPs"].plot(kind='bar', figsize=(10,5))
plt.title('DCP Proportions Across Time for Each Sample')
plt.xlabel('Sample')
plt.ylabel('DCP Proportion')
plt.xticks(rotation=45, ha='right')
plt.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.15),
    ncol =6,
    frameon=False
)
plt.tight_layout()
plt.show()  

In [ ]:
dcp_proportions.drop(columns=['Pre-sort', 'Sorted'], inplace=True)


In [ ]:
dcp_proportions

In [ ]:
# Plot bar chart of DCP proportions across time for each sample
plt.figure(figsize=(10, 6))
dcp_proportions.plot(kind='bar', figsize=(10,5))
plt.title('DCP Proportions Across Time for Each Sample')
plt.xlabel('Sample')
plt.ylabel('DCP Proportion')
plt.xticks(rotation=45, ha='right')
plt.legend(title="", frameon=False, ncol=2)
plt.tight_layout()
plt.show()  